# 04 — Generalization, Robustness, Uncertainty and Progressive Temporal Proof of Principle

**Questions:** How sensitive are held-out estimates to correlated frames and source identity? Which mistakes matter operationally? Does adding labelled historical video data improve classification on later frames?

The historical source-video bootstrap, failure analysis and grayscale evidence are accepted from archived results by default. An optional full historical rerun can recompute them. The supplementary **two-sequence progressive ResNet18 study was completed on 3 November 2026**. By default this notebook displays its committed eight-run summary; a separately enabled rerun invokes the tracked source functions and measures fresh results. Historical archived tables are labelled as such, not presented as newly computed values.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage
import src as mr
from src.temporal import run_progressive_temporal
ROOT=Path.cwd().resolve()
settings=mr.load_settings(ROOT)
settings['cache_resized_rgb'] = True  # Lossless 224x224 RGB cache across epochs
settings['progressive_batch_size'] = 4  # Conservative batch size for RTX 3050 Ti (4GB)
settings['log_progress_to_wandb'] = True  # Required: live charts for eight new runs
settings['progressive_wandb_project'] = 'monreader-progressive-temporal'
# Do not alter the research image resolution, held-out cohort, or decision threshold.
device=mr.setup(settings['seed'])
print('Resized RGB caching:', settings['cache_resized_rgb'])
print('W&B project:', settings['progressive_wandb_project'])
print('New progressive experiment batch size:', settings['progressive_batch_size'])
print('Device:',device,'| Source bootstrap resamples:',settings['bootstrap_resamples'])
# Setting RERUN_HISTORICAL_MODELS = False uses the accepted historical bootstrap,
# error and grayscale result tables, without repeating the old model runs.
# Setting RERUN_HISTORICAL_MODELS = True recomputes those diagnostics from
# the newly generated prediction CSVs (requires the optional full model rerun).
RERUN_HISTORICAL_MODELS = False
# All eight progressive T/ST runs were completed on 2026-11-03.
# Setting RUN_NEW_PROGRESSIVE_TEMPORAL = False displays their committed summary;
# set True ONLY to explicitly launch/resume new GPU training + W&B logging.
RUN_NEW_PROGRESSIVE_TEMPORAL = False
print('Historical re-evaluation:', RERUN_HISTORICAL_MODELS,
      '| New progressive T/ST study:', RUN_NEW_PROGRESSIVE_TEMPORAL)


## ResNet18 uncertainty across all four regimes

The difference between a score and its uncertainty matters. This cell loads frozen test predictions from Notebook 03, resamples **source groups with replacement**, recalculates Macro F1 and saves numerical intervals and a reproducible figure.

In [ ]:
if RERUN_HISTORICAL_MODELS:
    bootstrap_rows=[];all_draws={};error_rows=[]
    for regime in mr.REGIMES:
        p=pd.read_csv(Path(settings['out'])/'resnet18'/regime/'heldout_test_predictions.csv')
        r,draws=mr.source_bootstrap(p,seed=settings['seed'],reps=settings['bootstrap_resamples'])
        bootstrap_rows.append(dict(regime=regime,**r));all_draws[regime]=draws
        error_rows.append(dict(regime=regime,**mr.classify_errors(p)))
    bootstrap=pd.DataFrame(bootstrap_rows);errors=pd.DataFrame(error_rows)
    bootstrap.to_csv(Path(settings['out'])/'bootstrap_resnet18.csv',index=False)
    errors.to_csv(Path(settings['out'])/'resnet18_error_summary.csv',index=False)
    display(bootstrap);display(errors)
else:
    print("Original ResNet18 bootstrap and error evidence retained in submitted results; no historical retraining.")


In [ ]:
if RERUN_HISTORICAL_MODELS:
    plot_bootstrap=bootstrap[['regime','point_macro_f1','ci_lower','ci_upper']].rename(columns={'point_macro_f1':'point_estimate'})
else:
    archived=mr.read_archived_result_table(ROOT/'submission'/'results'/'bootstrap_summary.csv')
    plot_bootstrap=archived[archived.bootstrap_level=='source_video'][['regime','point_estimate','ci_lower','ci_upper']].copy()
    print('Figure represents the accepted historical source-video intervals.')
fig,ax=plt.subplots(figsize=(9,5),layout='constrained')
plot_bootstrap=plot_bootstrap.set_index('regime').loc[list(mr.REGIMES)].reset_index()
x=np.arange(len(plot_bootstrap))
lower=(plot_bootstrap.point_estimate-plot_bootstrap.ci_lower).clip(lower=0)
upper=(plot_bootstrap.ci_upper-plot_bootstrap.point_estimate).clip(lower=0)
ax.errorbar(x,plot_bootstrap.point_estimate,yerr=np.vstack([lower,upper]),fmt='o',capsize=6,elinewidth=2)
ax.set_xticks(x,plot_bootstrap.regime)
ax.set_ylim(0,1.04)
ax.set(ylabel='Held-out Macro F1',title='ResNet18 — source-video bootstrap 95% CI')
ax.grid(axis='y',alpha=.2)
p=Path(settings['out'])/'figures'/'resnet18_source_video_bootstrap.png'
p.parent.mkdir(parents=True,exist_ok=True)
fig.savefig(p,dpi=165,bbox_inches='tight')
plt.show()


## Paired RGB-versus-grayscale source bootstrap

The original paired-source bootstrap comparison is accepted historical evidence. With `RERUN_HISTORICAL_MODELS = False`, Notebook 04 displays the archived tables instead of recomputing predictions. An optional full rerun joins RGB and grayscale predictions on **identical sample IDs**, resamples source groups as paired units, and estimates the grayscale-minus-RGB held-out Macro F1 difference. A confidence interval containing zero does not establish equivalence.


In [ ]:
if RERUN_HISTORICAL_MODELS:
    color_rows=[]
    for regime in ('O','ST'):
        rgb=pd.read_csv(Path(settings['out'])/'scratch'/regime/'heldout_test_predictions.csv')
        gray=pd.read_csv(Path(settings['out'])/'grayscale'/regime/'heldout_test_predictions.csv')
        r=mr.paired_source_bootstrap(rgb,gray,seed=settings['seed'],reps=settings['bootstrap_resamples'])
        color_rows.append(dict(regime=regime,**r))
    color=pd.DataFrame(color_rows);color.to_csv(Path(settings['out'])/'paired_color_bootstrap.csv',index=False)
    display(color)
else:
    print('Accepted historical evidence is displayed in the submitted tables above.')


## Failure modes and confidence

A **false negative** is a real `flip` frame predicted `notflip`; a **false positive** is a real `notflip` frame predicted `flip`. A high-confidence error has a predicted-class probability of at least 0.90. In historical report mode the counts and plot come from the frozen submitted results, and in optional recomputation mode they come from newly generated per-frame predictions. Neither mode assumes that all future experiments will reproduce the historical all-false-negative pattern in T and ST.


In [ ]:
if RERUN_HISTORICAL_MODELS:
    fig,ax=plt.subplots(figsize=(8,4))
    x=np.arange(4);a=ax.bar(x-.20,errors.fp,width=.4,label='False positive');b=ax.bar(x+.20,errors.fn,width=.4,label='False negative')
    ax.set_xticks(x,errors.regime);ax.set(ylabel='Errors',title='ResNet18 Held-out False Positives and False Negatives');ax.legend()
    fig.tight_layout();path=Path(settings['out'])/'figures'/'resnet18_failure_analysis.png';fig.savefig(path,dpi=150);plt.show()
    print('Notebook 04 completed without manually supplied performance numbers.')
else:
    print('Accepted historical evidence is displayed in the submitted tables above.')


## Original generalization and uncertainty evidence — reference only

The archived bootstrap and failure tables preserve the earlier study. The new source-level bootstrap and progressive temporal evaluation remain separate experiments.

In [ ]:
historical_bootstrap=mr.read_archived_result_table(ROOT/'submission'/'results'/'bootstrap_summary.csv')
historical_errors=mr.read_archived_result_table(ROOT/'submission'/'results'/'final_error_summary.csv')
display(historical_bootstrap[['regime','bootstrap_level','point_estimate','ci_lower','ci_upper','n_resamples']])
display(historical_errors)

## Progressive T/ST experiment — completed methods and measured outputs

**Practical objective.** Estimate whether more chronologically available, **labelled** historical frames improve frame-wise page-flip classification, and whether performance differs for increasingly distant future image positions.

**Model and procedure.** Start two ImageNet-pretrained ResNet18 models: **T**, training on early frames of each familiar source; and **ST**, training on early frames from known sources only, with the **original frozen 24-source ST holdout** entirely withheld (489 early-context and 121 late-test frames in the historical 80/20 split). Train at the 25% checkpoint with the established two-stage configuration, then **continue fine-tuning the same model** as prefixes grow to 50%, 75% and 90%. No new Bayesian hyperparameter sweep is performed. A chronological validation tail within each available prefix supports checkpoint selection. The evaluation suffix is never used for that checkpoint's optimization. The 24 unseen ST sources are identical to the original reported ST holdout. The progressive 25/50/75% test suffixes may contain images that belonged to the 489 early-context-only frames under the historic 80/20 definition; these are **new secondary evaluations only**, not part of the original 121-frame ST score.

**Complementary analyses.** At each checkpoint, compute (1) one **overall future Macro F1** over all remaining frames, and (2) **horizon-window Macro F1 and flip recall**, pooling equivalent future positions across sources in three-frame windows. A window is reportable only with at least 20 frames per class and at least eight sources per class. Report class support and source counts, with no score for unsupported windows. To compare training coverage on the same examples, also evaluate the common final 10% of every eligible recording.

**Interpretation restrictions.** T tests later frames of familiar sources. ST tests later positions in **unseen recordings**: this is not decay since the model last observed the tested source. Frame numbers do not provide known time intervals or dates. These are retrospective supervised model updates that require labels; they do **not** establish online learning, real-time streaming efficiency, or forecasting of a future event before its frame is observed.

The cell below produces the measured tables, predictions, checkpoints, histories and plots, stored independently of the original T/ST model comparison. Figures are not available until it has run.


**Execution reliability:** The first run builds a losslessly resized PNG cache, which reduces repeated image decoding in later epochs. Training prints batch/epoch progress and saves the current stage's optimizer/RNG/validation-best checkpoint after each completed epoch, as well as a stage-level checkpoint after training. Runs interrupted before this revised checkpointing code cannot retroactively recover their unfinished epoch. Resume requires executing with the same split and protocol; mismatches are rejected. CPU-only training can still be slow. Do not run the progressive cell until the hardware decision is made.


## Eight completed progressive GPU/W&B runs — tracking and safe reruns

The supplementary experiment completed **eight distinct W&B runs** on 3 November 2026: four T and four ST checkpoints at 25%, 50%, 75% and 90% observed-frame coverage. These are not repetitions of the 120 accepted historical Scratch CNN qualifying trials.

**Default report mode (`RUN_NEW_PROGRESSIVE_TEMPORAL = False`)** reads the committed eight-run results table and makes no GPU calls or W&B updates. Set the switch to `True` only when an intentional new experiment or compatible checkpoint resume is needed, with the original image dataset, CUDA environment, W&B login, and sufficient runtime.

When a new run is enabled, per-epoch training/validation losses, validation Macro F1, accuracy and balanced accuracy, checkpoint stages, and final held-out results are logged to the separate W&B project. An interrupted run resumes only under the same experiment fingerprint. This is a new and explicitly labelled experiment: any regenerated scores must be compared to the completed 3 November evidence, not substituted for its observations or the original 121-frame historical ST benchmark.


In [ ]:
manifest=pd.read_csv(Path(settings['out'])/'canonical_manifest.csv')
original_st_split=Path(settings['out'])/'splits'/'ST_split.csv'
if RUN_NEW_PROGRESSIVE_TEMPORAL:
    overall_future,horizon_windows=run_progressive_temporal(
        manifest,settings,device,original_st_split
    )
    display(overall_future)
    display(horizon_windows[['regime','coverage_percent','start_horizon','end_horizon',
                             'n_frames','n_flip','n_notflip','n_sources','reportable',
                             'macro_f1','flip_recall']])
    for regime in ('T','ST'):
        folder=Path(settings['out'])/'progressive_temporal'/'figures'
        display(IPyImage(filename=str(folder/f'{regime}_overall_future.png')))
        display(IPyImage(filename=str(folder/f'{regime}_horizon_macro_f1.png')))
else:
    archived_progressive=pd.read_csv(ROOT/'submission/results/progressive_temporal_completed_8_runs.csv')
    assert len(archived_progressive)==8, 'Expected the eight completed T/ST run records'
    display(archived_progressive[['regime','coverage_percent','n_train','n_validation',
        'n_future','future_macro_f1','future_flip_recall','matched_macro_f1',
        'matched_flip_recall','false_negatives','false_positives']])
    print('Showing the completed 2026-11-03 runs; no new GPU training or W&B run launched.')


### Completed results interpretation — 3 November 2026

The completed supplementary study recorded eight GPU/W&B runs. For fixed final-10% evaluation cohorts, **T Macro F1** increased from **0.338274 to 0.730368** (327 frames across 117 familiar sources), and **ST Macro F1** from **0.346535 to 0.653066** (66 frames across 24 frozen unseen sources), between the 25% and 90% checkpoints. Intermediate trajectories were **not monotonic**.

At 90% coverage, T still missed **80** true flips; ST missed **18** true flips. The full remaining-future suffix decreases with coverage, so its changing-population scores are not directly comparable as an isolated treatment effect. Three-frame future-position windows are interpreted only when their per-class sample and source support passes the specified minimums.

The completed measurements are available in the committed `submission/results/progressive_temporal_completed_8_runs.csv` and the detailed provenance document `docs/PROGRESSIVE_TEMPORAL_FINAL_RESULTS_2026-11-03.md`. The notebook never uses historical ST's 489 context frames for model training or validation, although a new progressive ST evaluation suffix may score them. Frame positions do not establish elapsed-time drift, automatic online learning, or a real-time event prediction capability.
